In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "distilgpt2" #distilgpt2: a smaller and faster version of GPT-2

# Load the tokenizer used by the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Vocabulary size of {model_name} = {tokenizer.vocab_size}")

In [ ]:
# Set the prompt
prompt_text = "Though Fullerton is the largest city in the state of"

# Use the tokenizer to convert the prompt to tokens
inputs = tokenizer(prompt_text, return_tensors="pt")

# The 'input_ids' are the integer ids of the tokens
print("Prompt text:", prompt_text)
print("Token IDs:", inputs["input_ids"])

# Decode each token ID back to its string representation
tokens = [tokenizer.decode(token_id) for token_id in inputs["input_ids"][0].tolist()]
print("Tokens:", tokens)

In [ ]:
# Load the model
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
with torch.no_grad(): # disable gradient calculations as we are not training the model

    outputs = model(inputs["input_ids"]) # Get the model output in logits

    next_token_logits = outputs.logits[:, -1, :] # get the logits for only the last token in the input sequence

    # Convert logits into probabilities using the softmax function
    probabilities = torch.nn.functional.softmax(next_token_logits, dim=-1)

    # Find the token ID with the highest probability
    most_likely_next_token_id = torch.argmax(probabilities).item()

print(f"The most likely next token's id is: {most_likely_next_token_id}")
print(f"This token is: '{tokenizer.decode(most_likely_next_token_id)}'")
print(f"Highest probability is: '{torch.max(probabilities)}'")

In [ ]:
# generate tokens one after the other in a loop
generated_ids = inputs["input_ids"]

print("Starting prompt tokens:")
print(tokenizer.decode(generated_ids[0]))

# Generate 5 tokens, one token at a time
for _ in range(15):
    with torch.no_grad():
        outputs = model(generated_ids)
        next_token_logits = outputs.logits[:, -1, :]
        next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1) # convert to tensor

    # Append the predicted token_id to input sequence
    generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

    # Print the generated token
    print(tokenizer.decode(next_token_id[0]), end="")

In [ ]:
# generate multiple tokens using the built-in generate()
output_ids = model.generate(
    **inputs, max_length=50, pad_token_id=tokenizer.eos_token_id  # EOS: End-of-Sequence
)

# Decode the entire sequence of token IDs into a single string
generated_text = tokenizer.decode(output_ids[0])

print("Generated text:")
print(generated_text)